## Ingest raw data

In [0]:
raw_df = spark.read.csv("/Volumes/workspace/amazon/amazondata/Sales_data/", header=True, inferSchema=True)
display(raw_df)


## Clean and Validate Data with PySpark

In [0]:
from pyspark.sql.functions import col, to_date

clean_df = raw_df.withColumn("Date", to_date(col("Date"), "dd-MM-yyyy"))

clean_df = clean_df.dropna(subset=["Date", "Price", "Quantity"])

clean_df = clean_df.filter((col("Price") > 0) & (col("Quantity") > 0))
display(clean_df)

## monthly sales by region

In [0]:
from pyspark.sql.functions import year, month, sum, desc

agg_df = clean_df.withColumn("Year", year("Date")).withColumn("Month", month("Date"))

monthly_sales_region = agg_df.groupBy("Year", "Month", "Customer Location") \
                            .agg(sum("Total Sales").alias("Monthly Sales")) \
                            .orderBy("Year", "Month", "Customer Location")

display(monthly_sales_region)

### withColumnRenamed columns

In [0]:
monthly_sales_region = monthly_sales_region.withColumnRenamed("Customer Location", "Customer_Location").withColumnRenamed("Monthly Sales", "Monthly_Sales")

## Top 10 customers by profit

In [0]:
top_customers = clean_df.groupBy("Customer Name") \
                        .agg(sum("Total Sales").alias("Total Profit")) \
                        .orderBy(desc("Total Profit")) \
                        .limit(10)

display(top_customers)

## Category-wise average discount

In [0]:
from pyspark.sql.functions import avg

category_avg_price = clean_df.groupBy("Category") \
                             .agg(avg("Price").alias("Average Price"))

display(category_avg_price)

# reporting quries

### temporary view

In [0]:
clean_df.createOrReplaceTempView("sales_data")

### sales > 1000 (example: 1000)

In [0]:
%sql
SELECT * FROM sales_data WHERE `Total Sales` > 1000

### categories with total sales above threshold

In [0]:
%sql
SELECT Category, SUM(`Total Sales`) as TotalSales
FROM sales_data
GROUP BY Category
HAVING TotalSales > 2000

### Count number of completed orders by Payment Method

In [0]:
%sql
SELECT `Payment Method`, COUNT(*) as CompletedOrders
FROM sales_data
WHERE Status = 'Completed'
GROUP BY `Payment Method`
ORDER BY CompletedOrders DESC

### Saving file in parquet format

In [0]:
monthly_sales_region.write.mode("overwrite") \
                   .partitionBy("Year", "Month") \
                   .parquet("/Volumes/workspace/amazon_output_data/sales_data_table/monthly_sales_by_region")

In [0]:
# dbutils.fs.rm("/Volumes/workspace/amazon/amazondata/Sales_data/output_file/monthly_sales_by_region", recurse=True)

### Save tables in Databricks (managed tables)

In [0]:
monthly_sales_region.write.format("delta").mode("overwrite").saveAsTable("monthly_sales_by_region")

In [0]:
%sql
select * from monthly_sales_by_region;

# section 2


### Day 1: Foundations of PySpark and Data Ingestion  
**Learning Goals:**
- Understand what PySpark is and why it's useful.
- Set up Databricks (or local PySpark) environment.
- Learn how to read CSV files into DataFrames.

**Tools/Concepts:**
- PySpark basics (`SparkSession`, `DataFrame`)
- Navigating Databricks notebook
- Reading data from CSV

**Hands-on Lab:**
- Load the sample sales data CSV into a PySpark DataFrame.
- Display and explore the first few rows.

**Mini-Assessment:**
- **Quiz:** What’s the difference between a Spark DataFrame and a Pandas DataFrame?
- **Task:** Show you can load a CSV and print the schema.

---

### Day 2: Data Cleaning and Aggregation with PySpark  
**Learning Goals:**
- Learn to clean and validate data (handle missing values, types).
- Understand basic DataFrame operations: `filter`, `groupBy`, `agg`.
- Practice aggregation for simple analytics.

**Tools/Concepts:**
- PySpark SQL functions (`withColumn`, `dropna`, `filter`)
- Aggregation (`groupBy`, `sum`, `avg`)
- Data types and type casting

**Hands-on Lab:**
- Clean the sales data:
  - Convert ‘Date’ to date type.
  - Remove invalid entries (e.g., negative price/quantity).
  - Calculate monthly sales by region and category average price.

**Mini-Assessment:**
- **Quiz:** How do you handle nulls or bad data in PySpark?
- **Task:** Write a PySpark command to get top 3 customer locations by total sales.

---

### Day 3: Analytics, Reporting, and Sharing Results  
**Learning Goals:**
- Run analytical queries with Spark SQL.
- Save output data (Parquet/Delta/CSV).
- Practice sharing insights/reporting.

**Tools/Concepts:**
- Spark SQL (`createOrReplaceTempView`, SQL `SELECT` queries)
- Saving DataFrames to Parquet/Delta
- Visualizing/querying output in notebook

**Hands-on Lab:**
- Register cleaned DataFrame as a temp view.
- Run queries for:
  - Orders with sales above a threshold.
  - Top customers and regions.
- Save one result as a Parquet file.

**Mini-Assessment:**
- **Mini Challenge:** List the top selling product categories and save results.
- **Short Reflection:** What insights did you find from the data?


# Section 3


Hi **Student Name**,

First, I want to recognize your enthusiasm for getting results with Spark—taking action is a big part of learning data engineering. However, I noticed that your code is heavily hardcoded and lacks modular structure. **Modular code** is essential in real-world projects: it makes your logic easier to read, debug, and (most importantly) re-use as your pipeline grows. For example, creating functions for data cleaning or aggregation allows you to test, maintain, and adapt your code much more efficiently than repeating similar logic in multiple places.

---

### About your point on SQL:
It’s actually **still a core skill** for data engineers—**not at all outdated!**  
SQL is the backbone for querying, transforming, and summarizing data in almost every major platform—whether on traditional databases or within Spark itself (via **Spark SQL**).

- Filtering top customers  
- Joining sales to product tables  
- Doing complex `GROUP BY` operations  

These are often more **readable and concise** in SQL than with nested PySpark transformations. Also, most enterprise teams rely on SQL for analytics and reporting due to its **declarative style** and ease of optimization.

---

### 10-Day Skill Building Plan

**Day 1-2:** Refactor your current Spark job into **modular functions** (e.g., one for reading data, one for cleaning).

**Day 3:** Read about **why modular code matters**—explore real-life examples on GitHub.

**Day 4-5:** Take an online **SQL refresher** (LeetCode / Mode Analytics) and practice running SQL on sample datasets.

**Day 6:** Rewrite one Spark data transformation using **Spark SQL** instead of PySpark DataFrame code.

**Day 7:** Read about real-world data pipelines; notice how **reusable functions simplify** large projects.

**Day 8:** Pair up with a peer or mentor to **review/refactor each other’s code** for structure and clarity.

**Day 9:** Work on a small challenge: read a CSV, clean it, perform an aggregation, and save results —  
all with **modular code** and at least one **SQL query**.

**Day 10:** Reflect and document what you learned. Note improvements in:
- Code readability  
- Debugging  
- Performance

# Section 4


### SECTION 4: Quick Conceptual Check

**1. When would you choose Spark over Pandas?**  
When working with large datasets (in GBs or TBs) that don’t fit into memory, we use Spark. Pandas is better for small, in-memory data.

---

**2. What is a broadcast join in Spark and when should it be used?**  
A broadcast join sends a small table to all worker nodes to avoid shuffling. Use it when one table is small and the other is large.

---

**3. Difference between LEFT JOIN and INNER JOIN with an example:**  
- **LEFT JOIN** returns all records from the left table and matching rows from the right.  
- **INNER JOIN** returns only matching rows from both tables.  
Example: Joining customers with orders.  
- LEFT JOIN: Show all customers, even if no orders.  
- INNER JOIN: Show only customers who placed orders.

---

**4. What is a DAG in Airflow and how do you monitor it?**  
A DAG (Directed Acyclic Graph) is a workflow of tasks with dependencies.  
We monitor it using the Airflow UI — it shows task status (success, failed, etc.).

---

**5. How would you explain “partitioning” and “shuffling” to a beginner?**  
- **Partitioning** is dividing data into chunks so tasks can run in parallel.  
- **Shuffling** is moving data across partitions (nodes), usually during joins or aggregations.
